# Momentum: Can Past Winners Predict Future Winners?

**Research question.** Do stocks with the highest returns over the previous twelve months, excluding the most recent month, continue to outperform over the following month?

This notebook runs the analysis in `momentum_strategy.py` step by step. Every number and chart below is generated from data downloaded when the notebook runs.

In [ ]:
import pandas as pd
import momentum_strategy as ms

pd.set_option("display.float_format", "{:.4f}".format)
%matplotlib inline

## 1. Data

Daily adjusted closing prices for a fixed universe of 19 large-cap US stocks and SPY come from Yahoo Finance via `yfinance`. The risk-free rate is the 13-week T-bill yield (`^IRX`). Prices are converted to month-end values, and any incomplete final month is dropped.

In [ ]:
prices = ms.download_prices(ms.TICKERS + [ms.BENCHMARK])
rf = ms.download_risk_free()
monthly_prices = ms.to_monthly(prices)

print(f"{monthly_prices.index[0]:%b %Y} to {monthly_prices.index[-1]:%b %Y}, "
      f"{monthly_prices.shape[1]} series")
monthly_prices.tail()

## 2. Method

* **Signal.** For holding month *t*, the signal is each stock's cumulative return over months *t*−12 to *t*−2. Month *t*−1 is skipped to separate medium-term momentum from short-term reversal.
* **Portfolio.** At the close of month *t*−1, the top 20% of stocks by signal (3 of 19) are bought, equally weighted, held for month *t*, and rebalanced monthly.
* **Costs.** Net returns deduct 10 bp per dollar traded. Turnover allows for weights drifting during the month.
* **Comparisons.**
  * The equal-weighted portfolio of the same universe isolates the value of ranking by momentum.
  * SPY represents the broad market.
  * The winners-minus-losers (WML) spread, top 20% minus bottom 20%, is the standard test of the momentum effect.

In [ ]:
out = ms.analyse(monthly_prices, rf)
results = out["results"]
print(f"Backtest: {results.index[0]:%b %Y} to {results.index[-1]:%b %Y} "
      f"({len(results)} months)")
results.head()

## 3. Performance

In [ ]:
out["stats"]

In [ ]:
print(f"Average one-way monthly turnover: {out['avg_turnover']:.1%}")

## 4. Statistical significance

The t-statistics test whether each mean monthly return or spread differs from zero; |t| > 2 is roughly significant at the 5% level. With around ten years of monthly data, these tests have limited power.

The CAPM regressions estimate each portfolio's alpha and beta against SPY, using classical standard errors.

In [ ]:
out["tests"]

In [ ]:
out["capm"]

## 5. Charts

In [ ]:
paths = ms.make_figures(results, show=True)

## 6. Holdings

These are the stocks held in the most recent months, together with how often each stock appears in the winner portfolio across the full sample.

In [ ]:
h = out["holdings"]
display(h[h["date"] >= h["date"].unique()[-6]].pivot(index="date", columns="rank", values="ticker"))
h["ticker"].value_counts()

## 7. Robustness

The strategy is re-run with lookbacks of 3, 6, 9 and 12 months and with the top 20% or top 30% of stocks. A result that holds only under one specification should be treated with caution.

In [ ]:
grid = ms.robustness_grid(monthly_prices, rf)
grid

## 8. Limitations

* **Selection bias.** The universe consists of companies that are large and successful *today*, chosen with hindsight. This biases all long portfolios upward relative to SPY. The equal-weight comparison and the WML spread are more informative than the comparison with SPY.
* **Small cross-section.** With 19 stocks, each portfolio holds only three. Results are therefore noisy and dominated by a few names.
* **Short sample.** About ten years of data gives low statistical power, and the sample includes the 2020 shock and the post-2022 rotation.
* **Simplified frictions.** The backtest models a flat cost per dollar traded but ignores taxes, market impact, short-selling costs for the WML leg, and delistings.
* **Data quality.** `yfinance` is an unofficial interface to Yahoo Finance. The data are adequate for an educational study but are not institutional quality.

This is an educational research project, not investment advice.

In [ ]:
# Save tables for the write-up
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)
results.to_csv("outputs/monthly_returns.csv")
out["holdings"].to_csv("outputs/holdings.csv", index=False)
out["stats"].to_csv("outputs/performance_stats.csv")
out["tests"].to_csv("outputs/significance_tests.csv")
out["capm"].to_csv("outputs/capm_regressions.csv")
grid.to_csv("outputs/robustness_grid.csv", index=False)